In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, r2_score
from sklearn import datasets

# implementation of SVC

In [11]:

data = datasets.load_iris(as_frame = True).frame
data.info()
data.shape
x = data.drop("target", axis = 1)
y = data["target"]
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=42, test_size=0.3, stratify = y) # stratify= y is used for imbalance classes
ss = StandardScaler()
x_train_scaled = ss.fit_transform(x_train)
x_test_scaled = ss.transform(x_test)
svc = SVC(kernel="linear") # with linear kernel
svc.fit(x_train_scaled, y_train)
pred_svc = svc.predict(x_test_scaled)
print("The accuracy of the model is", accuracy_score(y_test,pred_svc))

# for polynomial  

svc = SVC(kernel="poly")
svc.fit(x_train_scaled, y_train)
pred_svc = svc.predict(x_test_scaled)
print("The accuracy of the model is ", accuracy_score(y_test,pred_svc))#

# for sigmoid kernel 

svc = SVC(kernel="sigmoid")
svc.fit(x_train_scaled, y_train)
pred_svc = svc.predict(x_test_scaled)
print("The accuracy of the model is", accuracy_score(y_test,pred_svc))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB
The accuracy of the model is 0.9111111111111111
The accuracy of the model is  0.8666666666666667
The accuracy of the model is 0.9111111111111111


# implementation of SVR (Support Vector Regressor)

In [12]:
from sklearn.svm import SVR
df = datasets.load_diabetes(as_frame=True).frame

In [22]:
x = df.drop("target", axis = 1)
y = df["target"]
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=42, test_size=0.3) # stratify= y is used for imbalance classes
ss = StandardScaler()
y_train_scaled = ss.fit_transform(y_train.values.reshape(-1,1)).ravel()
y_test_sclaed = ss.fit_transform(y_test.values.reshape(-1,1)).ravel()
model = SVR()
model.fit(x_train, y_train_scaled)
pred_svr = model.predict(x_test)
print("r2 value  : ", r2_score(y_test_sclaed, pred_svr))

r2 value  :  0.4808287850481272


# Hyper-Perameters Tuning using GridSearchCV

In [26]:
from sklearn.model_selection import GridSearchCV
prag_grid = {
    "C" : [1,2,5,10,20,100],
    "kernel" : ["linear", "rbf"],
    "epsilon" : [0.01,0.02,0.1,1,2]
    
}
grid_search = GridSearchCV(model, prag_grid, scoring="r2", cv=5)
grid_search.fit(x_train, y_train_scaled)
print(" The best parameters are", grid_search.best_params_)

 The best parameters are {'C': 20, 'epsilon': 0.1, 'kernel': 'linear'}


In [31]:
# best model
best_model = SVR(C=20, epsilon=0.1, kernel="linear")
best_model.fit(x_train, y_train_scaled)
y_test_pred_scaled= best_model.predict(x_test)
y_train_pred_scaled= best_model.predict(x_train)
print("r2 value  : ", r2_score(y_test_sclaed, y_test_pred_scaled))
print("r2 value  : ", r2_score(y_train_scaled, y_train_pred_scaled))


r2 value  :  0.4519740186976826
r2 value  :  0.5148721777019414


# RANDOM FOREST


In [32]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [33]:
titanic = sns.load_dataset("titanic")

features = ["pclass", "sex", "fare", "embarked", "age"]
target = ["survived"]

# handle missing data
imp_median = SimpleImputer(strategy="median")
titanic[["age"]] = imp_median.fit_transform(titanic[["age"]])

imp_freq = SimpleImputer(strategy="most_frequent")
titanic[["embarked"]] = imp_freq.fit_transform(titanic[["embarked"]])

# encode
le = LabelEncoder()

titanic["sex"] = le.fit_transform(titanic["sex"])
titanic["embarked"] = le.fit_transform(titanic["embarked"])

X = titanic[features]
y = titanic["survived"]

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state=42
)

In [34]:
# Decision Tree
model = DecisionTreeClassifier(max_depth=4)

model.fit(X_train, y_train)

y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

print("Training accuracy: ", accuracy_score(y_train, y_pred_train)*100, "%")
print("Testing accuracy: ", accuracy_score(y_test, y_pred_test)*100, "%")
# classic case of overfitting

Training accuracy:  84.75120385232745 %
Testing accuracy:  82.46268656716418 %


In [35]:
# NOW TO IMPLEMENT RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=501,
    oob_score=True,
    max_depth=4)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("OOB score: ", rf.oob_score_ * 100, "%")
print("testing accuracy: ", accuracy_score(y_test, y_pred) * 100, "%")

OOB score:  82.34349919743178 %
testing accuracy:  82.08955223880598 %


In [39]:
# Bagging Classifier
from sklearn.ensemble import BaggingClassifier
base_model = DecisionTreeClassifier()
BC = BaggingClassifier(
    base_model,
    n_estimators=201,)
BC.fit(X_train, y_train)

y_pred = BC.predict(X_test)

print("Accuracy", accuracy_score(y_test, y_pred) *100 , "%")
    

Accuracy 78.35820895522389 %


In [42]:
# Bagging Regression
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
base_model = DecisionTreeRegressor()
BR = BaggingRegressor(
    base_model,
    n_estimators=201,)
BR.fit(X_train, y_train)

y_pred = BR.predict(X_test)

print("R2", r2_score(y_test, y_pred) *100 , "%")

R2 35.16735893210575 %
